## Practice 2 for tests

In [ ]:
#Kernel anaconda pyton 3.13.9
import pandas as pd
import numpy as np

### Section 1: Object Oriented Programming (OOP)

#### Exercise 1.1 — The DataRecord Class
Problem:
Create a class DataRecord that represents a row in a dataset.

Requirements:

The constructor takes **kwargs (arbitrary column names and values).

Add a method get(column, default=None) that returns the value of a column, or the default if the column doesn't exist.

Add a method to_dict() that returns the internal data as a plain dictionary.

Add a __repr__ method that returns something like: DataRecord(id=1, name='Alice').

Add a class method from_csv_line(cls, line, columns) that takes a CSV string like "1,Alice,30" and a list of column names ['id', 'name', 'age'] and returns a DataRecord instance with proper types (convert numeric strings to int or float).

In [15]:
class DataRecord:
    def __init__(self,data):
        self.data = data
    def get(self, column, default=None):
        value=self.data.get(column, default)
        return value
    def to_dict(self):
        return self.data
    def __repr__(self):
        return f"DataRecord({self.data})"
    def _cast(value):
        """
        Convert a string to int, float if applicable
        """
        value=value.strip()
        try:
            return int(value)
        except ValueError:
            pass
        try:
            return float(value)
        except ValueError:
            return value

    @classmethod
    def from_csv_line(cls, line, columns):
        values = line.strip().split(',')
        data = dict(zip(columns, [cls._cast(v) for v in values]))
        return cls(data)


In [16]:
#test
r = DataRecord.from_csv_line("1,Alice,30,5.5", ['id', 'name', 'age', 'score'])
print(r)                    # DataRecord(id=1, name='Alice', age=30, score=5.5)
print(r.get('age'))         # 30
print(r.get('salary', 0))   # 0
print(r.to_dict())          # {'id': 1, 'name': 'Alice', 'age': 30, 'score': 5.5}

DataRecord({'id': 1, 'name': 'Alice', 'age': 30, 'score': 5.5})
30
0
{'id': 1, 'name': 'Alice', 'age': 30, 'score': 5.5}


#### Exercise 1.2 — Inheritance: Cleaner and Transformer
Problem:
Build a small pipeline of classes.

Create a base class Step with:

A method run(self, record: dict) -> dict that raises NotImplementedError.

A __call__ method that delegates to run (so instances are callable).

Create a subclass Cleaner that:

Takes a list of column names in its constructor.

Strips whitespace from string values and replaces empty strings with None.

Create a subclass Transformer that:

Takes a column name and a func (function).

Applies func to that column's value and returns the updated record.

Create a Pipeline class that holds a list of steps and runs them in order on each record.

In [8]:
class Step:
    def run(self, record: dict) -> dict:
        raise NotImplementedError("Subclasses must implement run()")

    def __call__(self, record: dict) -> dict:
        return self.run(record)


class Cleaner(Step):
    def __init__(self, columns):
        self.columns = columns

    def run(self, record):
        new_record = dict(record)
        for col in self.columns:
            if col in new_record and isinstance(new_record[col], str):
                value = new_record[col].strip()
                new_record[col] = value if value else None
        return new_record


class Transformer(Step):
    def __init__(self, column, func):
        self.column = column
        self.func = func

    def run(self, record):
        new_record = dict(record)
        if self.column in new_record and new_record[self.column] is not None:
            new_record[self.column] = self.func(new_record[self.column])
        return new_record


class Pipeline:
    def __init__(self, steps):
        self.steps = steps

    def run(self, records):
        result = []
        for record in records:
            for step in self.steps:
                record = step(record)
            result.append(record)
        return result


# Test
records = [
    {"name": "  Alice ", "age": 30, "score": 45.0},
    {"name": "", "age": 25, "score": 82.0},
]

pipe = Pipeline([
    Cleaner(['name']),
    Transformer('score', lambda x: x / 100),
])

for r in pipe.run(records):
    print(r)
# {'name': 'Alice', 'age': 30, 'score': 0.45}
# {'name': None, 'age': 25, 'score': 0.82}

{'name': 'Alice', 'age': 30, 'score': 0.45}
{'name': None, 'age': 25, 'score': 0.82}


In [25]:
class Step:
    #def __init__(self,data):
    #    self.data = data
    def run(self,record:dict) -> dict:
        raise NotImplementedError("Subclasses should implement this method")
    def __call__(self,record:dict)-> dict:
        return self.run(record)

class Cleaner(Step):
    def __init__(self,columns):
        self.columns = columns
    def run(self,record:dict)-> dict:
        for column in self.columns:
            if column in record and isinstance(record[column], str):
                record[column] = record[column].strip()
            if record[column] == '':
                record[column] = None
        return record
class Transformer(Step):
    def __init__(self,column,transform_func):
        self.column = column
        self.transform_func = transform_func

    def run(self,record:dict)-> dict:
        if self.column in record:
            record[self.column] = self.transform_func(record[self.column])
        return record
class Pipeline:
    def __init__(self,steps:list):
        self.steps = steps
    def run(self,records:list):
        for record in records:
            for step in self.steps:
                record = step(record)
            yield record

In [26]:
# Test
records = [
    {"name": "  Alice ", "age": 30, "score": 45.0},
    {"name": "", "age": 25, "score": 82.0},
]

pipe = Pipeline([
    Cleaner(['name']),
    Transformer('score', lambda x: x / 100),
])

for r in pipe.run(records):
    print(r)
# {'name': 'Alice', 'age': 30, 'score': 0.45}
# {'name': None, 'age': 25, 'score': 0.82}

{'name': 'Alice', 'age': 30, 'score': 0.45}
{'name': None, 'age': 25, 'score': 0.82}


### Section 2: List & Dictionaries

#### Exercise 2.1 — Log Analysis
Problem:
You have a list of log entries. Each entry is a dictionary like:

python
{"user": "alice", "action": "login", "status": "success"}
Write a function summarize(logs) that returns a dictionary:

text
{
  "alice": {"login": 3, "logout": 1},
  "bob":   {"login": 1, "purchase": 2}
}
Only count status == "success" entries. Skip entries with status != "success".

Bonus: return the top user by total actions.

In [ ]:
def summarize(logs):
    summary = {}
    for log in logs:
        if log["user"] not in summary:
            summary[log["user"]] = {}

        user = log["user"]
        action = log["action"]

        if action not in summary[user] and log["status"] == "success":  
            summary[user][action] = 0
        if log["status"] == "success":
            summary[user][action] += 1

    return summary

def top_user(summary):
    top_user = None
    max_actions = 0
    for user in summary:
        total_actions = sum(summary[user].values())
        if total_actions > max_actions:
            max_actions = total_actions
            top_user = user
    return top_user


In [43]:
summary = summarize(logs)
print(summary['alice']['login'])


for user in summary:
    totalactions=0
    for action in summary[user]:
        totalactions += summary[user][action]

    print(f"{user}: {totalactions} actions")

2
alice: 3 actions
bob: 3 actions


In [45]:

# Test
logs = [
    {"user": "alice", "action": "login", "status": "success"},
    {"user": "alice", "action": "login", "status": "success"},
    {"user": "alice", "action": "logout", "status": "success"},
    {"user": "alice", "action": "login", "status": "fail"},
    {"user": "bob", "action": "login", "status": "success"},
    {"user": "bob", "action": "purchase", "status": "success"},
    {"user": "bob", "action": "purchase", "status": "success"},
]

s = summarize(logs)
print(s)
# {'alice': {'login': 2, 'logout': 1}, 'bob': {'login': 1, 'purchase': 2}}
print(top_user(s))  # bob  (3 actions vs alice's 3 — tie, max returns first)

{'alice': {'login': 2, 'logout': 1}, 'bob': {'login': 1, 'purchase': 2}}
alice


In [ ]:


# Test
logs = [
    {"user": "alice", "action": "login", "status": "success"},
    {"user": "alice", "action": "login", "status": "success"},
    {"user": "alice", "action": "logout", "status": "success"},
    {"user": "alice", "action": "login", "status": "fail"},
    {"user": "bob", "action": "login", "status": "success"},
    {"user": "bob", "action": "purchase", "status": "success"},
    {"user": "bob", "action": "purchase", "status": "success"},
]

s = summarize(logs)
print(s)
# {'alice': {'login': 2, 'logout': 1}, 'bob': {'login': 1, 'purchase': 2}}
print(top_user(s))  # bob  (3 actions vs alice's 3 — tie, max returns first)

### Section 3: Pandas Data Transformation

#### Exercise 3.1 — CASE WHEN + JOIN
Problem:
You have two DataFrames:

python
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Carol", "Dan"],
    "country": ["US", "MX", "US", "CA"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 2, 3, 5],
    "amount": [50.0, 200.0, 30.0, 120.0, 75.0]
})
Tasks:

Left join orders onto customers (keep all customers, even those without orders).

Create a new column tier using CASE WHEN logic:

amount >= 100 → "high"

amount >= 50 → "medium"

amount < 50 → "low"

amount is NaN (no order) → "none"

Group by country and tier, and count orders.

In [46]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Carol", "Dan"],
    "country": ["US", "MX", "US", "CA"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 2, 3, 5],
    "amount": [50.0, 200.0, 30.0, 120.0, 75.0]
})

In [50]:
orders_customers_df=orders.merge(customers,right_on="customer_id",\
                                 left_on="customer_id",how="outer")
orders_customers_df.head(10)

,order_id,customer_id,amount,name,country
0,101.0,1,50.0,Alice,US
1,102.0,2,200.0,Bob,MX
2,103.0,2,30.0,Bob,MX
3,104.0,3,120.0,Carol,US
4,NaN,4,NaN,Dan,CA
5,105.0,5,75.0,NaN,NaN


In [65]:
answer=orders_customers_df.copy()
answer["tier"]=answer["amount"].apply(lambda x:  "high" if x>=100 else\
                                      "medium" if x>=50 else\
                                        "low" if x<50 else "none")

print(answer.head(20))

   order_id  customer_id  amount   name country    tier
0     101.0            1    50.0  Alice      US  medium
1     102.0            2   200.0    Bob      MX    high
2     103.0            2    30.0    Bob      MX     low
3     104.0            3   120.0  Carol      US    high
4       NaN            4     NaN    Dan      CA    none
5     105.0            5    75.0    NaN     NaN  medium


In [72]:
answer_gb=answer.groupby(["country","tier"])["order_id"]\
    .count().reset_index(name="count")
answer_gb.head(10)

,country,tier,count
0,CA,none,0
1,MX,high,1
2,MX,low,1
3,US,high,1
4,US,medium,1


In [87]:
answer_gb=answer.groupby(["country","tier"])\
    .agg(count=("order_id", "count"),total_amount=("amount", "sum")).reset_index()
answer_gb.head(10)

,country,tier,count,total_amount
0,CA,none,0,0.0
1,MX,high,1,200.0
2,MX,low,1,30.0
3,US,high,1,120.0
4,US,medium,1,50.0
